In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform

from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.inspection import permutation_importance

import xgboost as xgb

### How to Make a Custom Scorer

In [2]:
BrierScorer = make_scorer(brier_score_loss, greater_is_better=False, needs_proba=True)

### How to make a custom scorer
def rounded_RMSE(y_true, y_pred):
    rounded_preds = np.round(np.array(y_pred))
    return np.sqrt(np.mean((np.array(y_true) - rounded_preds) ** 2))

rounded_rmse_scorer = make_scorer(rounded_RMSE, greater_is_better=False)

c:\Users\seanv\Anaconda3_old\envs\gpt-env\lib\site-packages\sklearn\metrics\_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


### How to Make a CV Split by Year

In [3]:
import pandas as pd
import os

# Assume root directory is './TrainingData'
root_dir = './TrainingData'

TRAIN_list = []
LABELS_list = []
fold_indices = []

SALE_COLUMNS = ["Actual Loss Calculation", "Zero Balance Removal UPB", "Net Sales Proceeds", 
                "Delinquent Accrued Interest", "Expenses", "MI Recoveries", "Non MI Recoveries"]

MISC_COLUMNS_TO_DROP = ["Metropolitan Statistical Area (MSA) Or Metropolitan Division", 'Postal Code', 
                        'Distress Date', 'Default Flag', 'Zero Balance Code', 'Unresolved', 'Last Time Current',
                        'Property State', 'Current Loan Delinquency Status', 'Loan Sequence Number']
CAT_COLUMNS = []

fold = 0 

# Load in all files from TrainingData Folder
for year_folder in sorted(os.listdir(root_dir)):
    year_path = os.path.join(root_dir, year_folder)
    if os.path.isdir(year_path):
        for file in os.listdir(year_path):
            if file.endswith('.parquet'):
                file_path = os.path.join(year_path, file)
                df = pd.read_parquet(file_path)
                TRAIN_list.append(df.drop(columns=['Major Stress'])) 
                LABELS_list.append(df['Major Stress'])               
                fold_indices.extend([fold] * len(df))
        fold += 1

TRAIN = pd.concat(TRAIN_list).reset_index(drop=True)
TRAIN = TRAIN.drop(columns=SALE_COLUMNS)



In [7]:
TRAIN[["Metropolitan Statistical Area (MSA) Or Metropolitan Division"]].drop_duplicates().dropna().to_csv("msa.csv", index=False)

In [ ]:
TRAIN = TRAIN.drop(columns=MISC_COLUMNS_TO_DROP)
categorical_cols = TRAIN.select_dtypes(include=['object', 'category']).columns.tolist()

# Use below to add columns that we want dummies for but may not current be dtype categorical
# categorical_cols += CAT_COLUMNS

TRAIN = pd.get_dummies(TRAIN, columns=categorical_cols, drop_first=True)
LABELS = pd.concat(LABELS_list).reset_index(drop=True)
LABELS.fillna(0, inplace=True)

ps = PredefinedSplit(test_fold=fold_indices)



### How to Wrap XGB in a SKL Grid Search CVC
Scorer should be something like Brier Score or F1 (needs to account for the ridiculous imabalance and also should make sure probabilities are well calibrated)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from scipy.stats import uniform

state = 583
NumParamSearchIterations = 5


X_train, X_val, y_train, y_val = train_test_split(TRAIN, LABELS, test_size=0.2, random_state=state, stratify=LABELS)

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")

param_distributions = {
    'n_estimators': range(100, 200),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': range(5, 20),
    'subsample': uniform(0, 1),
    'colsample_bytree': uniform(0, 1)
}

hyperparameter_tuning = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=state, 
                      scale_pos_weight=470,
                      objective='binary:logistic',
                      eval_metric='logloss'),
    scoring='average_precision',  # <-- score using average precision
    param_distributions=param_distributions,
    n_iter=NumParamSearchIterations,
    cv=3,
    random_state=state,
    verbose=2
)

# Fit search on training split only
search = hyperparameter_tuning.fit(X_train, y_train)

print("\nBest Hyperparameters:", search.best_params_)
print("Best Average Precision Score (CV on train split):", search.best_score_)

# Evaluate best model on validation set
best_model = search.best_estimator_

y_val_pred_probs = best_model.predict_proba(X_val)[:, 1]
y_val_pred_labels = (y_val_pred_probs >= 0.5).astype(int)

# Calculate metrics
ap_score = average_precision_score(y_val, y_val_pred_probs)
precision = precision_score(y_val, y_val_pred_labels)
recall = recall_score(y_val, y_val_pred_labels)
f1 = f1_score(y_val, y_val_pred_labels)

print("\nValidation Metrics:")
print(f"Average Precision Score: {ap_score:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

# Get precision, recall, thresholds
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_pred_probs)

# Find threshold that gives best F1 score
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
print(f"\nBest threshold for max F1: {best_threshold:.4f}")
print(f"Precision at best threshold: {precisions[best_idx]:.4f}")
print(f"Recall at best threshold: {recalls[best_idx]:.4f}")
print(f"F1 at best threshold: {f1_scores[best_idx]:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_pred_probs)

plt.plot(recalls, precisions, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid()
plt.show()
